In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
silver_customers_df = spark.read.table("silver_customers")
silver_products_df = spark.read.table("silver_products")
silver_stores_df = spark.read.table("silver_stores")
silver_sales_df = spark.read.table("silver_sales")

StatementMeta(, df043938-5f7d-4ee1-99dc-73c71df1e18b, 3, Finished, Available, Finished, False)

In [11]:
from pyspark.sql.functions import monotonically_increasing_id, col

dim_customer_df = (
    silver_customers_df
    .select(
        "CustomerID",
        "FirstName",
        "LastName",
        "Email",
        "Phone",
        "City",
        "State",
        "Region",
        "SignupDate",
        "CustomerSegment"
    )
    .dropDuplicates(["CustomerID"])
    .withColumn(
        "CustomerKey",
        monotonically_increasing_id()
    )
)

StatementMeta(, 23642936-a065-41bf-b59a-4ea55a1fdb36, 13, Finished, Available, Finished, False)

In [3]:
display(dim_customer_df.limit(10))

StatementMeta(, df043938-5f7d-4ee1-99dc-73c71df1e18b, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 05469d5d-1b20-4962-8882-461dba4c2e00)

In [4]:
dim_customer_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_dim_customer")

StatementMeta(, df043938-5f7d-4ee1-99dc-73c71df1e18b, 6, Finished, Available, Finished, False)

In [5]:
from pyspark.sql.functions import monotonically_increasing_id

dim_product_df = (
    silver_products_df
    .select(
        "ProductID",
        "ProductName",
        "Category",
        "Brand",
        "UnitPrice",
        "UnitCost",
        "Supplier",
        "ActiveFlag"
    )
    .dropDuplicates(["ProductID"])
    .withColumn(
        "ProductKey",
        monotonically_increasing_id()
    )
)

StatementMeta(, df043938-5f7d-4ee1-99dc-73c71df1e18b, 7, Finished, Available, Finished, False)

In [1]:
display(dim_product_df.limit(10))

StatementMeta(, 23642936-a065-41bf-b59a-4ea55a1fdb36, 3, Finished, Available, Finished, False)

NameError: name 'dim_product_df' is not defined

In [5]:
from pyspark.sql.functions import monotonically_increasing_id

dim_product_df = (
    silver_products_df
    .select(
        "ProductID",
        "ProductName",
        "Category",
        "Brand",
        "UnitPrice",
        "UnitCost",
        "Supplier",
        "ActiveFlag"
    )
    .dropDuplicates(["ProductID"])
    .withColumn(
        "ProductKey",
        monotonically_increasing_id()
    )
)

StatementMeta(, 23642936-a065-41bf-b59a-4ea55a1fdb36, 7, Finished, Available, Finished, False)

In [3]:
silver_customers_df = spark.read.table("silver_customers")
silver_products_df = spark.read.table("silver_products")
silver_stores_df = spark.read.table("silver_stores")
silver_sales_df = spark.read.table("silver_sales")

StatementMeta(, 23642936-a065-41bf-b59a-4ea55a1fdb36, 5, Finished, Available, Finished, False)

In [6]:
dim_product_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_dim_product")

StatementMeta(, 23642936-a065-41bf-b59a-4ea55a1fdb36, 8, Finished, Available, Finished, False)

In [7]:
from pyspark.sql.functions import monotonically_increasing_id

dim_store_df = (
    silver_stores_df
    .select(
        "StoreID",
        "StoreName",
        "City",
        "State",
        "Region",
        "OpenDate",
        "StoreType"
    )
    .dropDuplicates(["StoreID"])
    .withColumn(
        "StoreKey",
        monotonically_increasing_id()
    )
)

StatementMeta(, 23642936-a065-41bf-b59a-4ea55a1fdb36, 9, Finished, Available, Finished, False)

In [8]:
display(dim_store_df.limit(10))

StatementMeta(, 23642936-a065-41bf-b59a-4ea55a1fdb36, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7c13c23b-967b-4d0f-8fd8-10d12e62d95c)

In [9]:
dim_store_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_dim_store")

StatementMeta(, 23642936-a065-41bf-b59a-4ea55a1fdb36, 11, Finished, Available, Finished, False)

In [12]:
fact_sales_df = (
    silver_sales_df.alias("s")

    .join(
        dim_customer_df.select("CustomerID", "CustomerKey").alias("c"),
        col("s.CustomerID") == col("c.CustomerID"),
        "left"
    )

    .join(
        dim_product_df.select("ProductID", "ProductKey").alias("p"),
        col("s.ProductID") == col("p.ProductID"),
        "left"
    )

    .join(
        dim_store_df.select("StoreID", "StoreKey").alias("st"),
        col("s.StoreID") == col("st.StoreID"),
        "left"
    )

    .join(
        dim_date_df.select("Date").alias("d"),
        to_date(col("s.OrderDateTime")) == col("d.Date"),
        "left"
    )

    .select(
        col("s.SalesID"),
        col("c.CustomerKey"),
        col("p.ProductKey"),
        col("st.StoreKey"),
        to_date(col("s.OrderDateTime")).alias("Date"),
        col("s.Quantity"),
        col("s.UnitPrice"),
        col("s.DiscountPct"),
        col("s.TotalAmount"),
        col("s.SalesChannel"),
        col("s.PaymentMethod")
    )
)

StatementMeta(, 23642936-a065-41bf-b59a-4ea55a1fdb36, 14, Finished, Available, Finished, False)

NameError: name 'dim_date_df' is not defined

In [14]:
from pyspark.sql.functions import col, to_date

dim_customer_df = spark.read.table("gold_dim_customer")
dim_product_df = spark.read.table("gold_dim_product")
dim_store_df = spark.read.table("gold_dim_store")
dim_date_df = spark.read.table("gold_dim_date")

silver_sales_df = spark.read.table("silver_sales")

StatementMeta(, 23642936-a065-41bf-b59a-4ea55a1fdb36, 16, Finished, Available, Finished, False)

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `gold_dim_date` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.;
'UnresolvedRelation [gold_dim_date], [], false


In [15]:
from pyspark.sql.functions import (
    col,
    to_date,
    year,
    month,
    dayofmonth,
    quarter,
    date_format
)

silver_sales_df = spark.read.table("silver_sales")

StatementMeta(, 23642936-a065-41bf-b59a-4ea55a1fdb36, 17, Finished, Available, Finished, False)

In [16]:
dim_date_df = (
    silver_sales_df
    .select(
        to_date(col("OrderDateTime")).alias("Date")
    )
    .dropDuplicates(["Date"])
    .withColumn("Year", year(col("Date")))
    .withColumn("MonthNumber", month(col("Date")))
    .withColumn("MonthName", date_format(col("Date"), "MMMM"))
    .withColumn("Day", dayofmonth(col("Date")))
    .withColumn("Quarter", quarter(col("Date")))
    .withColumn("DayName", date_format(col("Date"), "EEEE"))
)

StatementMeta(, 23642936-a065-41bf-b59a-4ea55a1fdb36, 18, Finished, Available, Finished, False)

In [17]:
display(dim_date_df.orderBy("Date").limit(10))

StatementMeta(, 23642936-a065-41bf-b59a-4ea55a1fdb36, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 55edffd8-20ba-4161-81fc-eb71c60cdb0c)

In [18]:
dim_date_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_dim_date")

StatementMeta(, 23642936-a065-41bf-b59a-4ea55a1fdb36, 20, Finished, Available, Finished, False)

In [19]:
dim_date_df = spark.read.table("gold_dim_date")

StatementMeta(, 23642936-a065-41bf-b59a-4ea55a1fdb36, 21, Finished, Available, Finished, False)

In [20]:
dim_date_df = spark.read.table("gold_dim_date")

StatementMeta(, 23642936-a065-41bf-b59a-4ea55a1fdb36, 22, Finished, Available, Finished, False)

In [21]:
from pyspark.sql.functions import col, to_date

dim_customer_df = spark.read.table("gold_dim_customer")
dim_product_df = spark.read.table("gold_dim_product")
dim_store_df = spark.read.table("gold_dim_store")
dim_date_df = spark.read.table("gold_dim_date")
silver_sales_df = spark.read.table("silver_sales")



StatementMeta(, 23642936-a065-41bf-b59a-4ea55a1fdb36, 23, Finished, Available, Finished, False)

In [22]:
fact_sales_df = (
    silver_sales_df.alias("s")

    .join(
        dim_customer_df.select("CustomerID", "CustomerKey").alias("c"),
        col("s.CustomerID") == col("c.CustomerID"),
        "left"
    )

    .join(
        dim_product_df.select("ProductID", "ProductKey").alias("p"),
        col("s.ProductID") == col("p.ProductID"),
        "left"
    )

    .join(
        dim_store_df.select("StoreID", "StoreKey").alias("st"),
        col("s.StoreID") == col("st.StoreID"),
        "left"
    )

    .join(
        dim_date_df.select("Date").alias("d"),
        to_date(col("s.OrderDateTime")) == col("d.Date"),
        "left"
    )

    .select(
        col("s.SalesID"),
        col("c.CustomerKey"),
        col("p.ProductKey"),
        col("st.StoreKey"),
        to_date(col("s.OrderDateTime")).alias("Date"),
        col("s.Quantity"),
        col("s.UnitPrice"),
        col("s.DiscountPct"),
        col("s.TotalAmount"),
        col("s.SalesChannel"),
        col("s.PaymentMethod")
    )
)

StatementMeta(, 23642936-a065-41bf-b59a-4ea55a1fdb36, 24, Finished, Available, Finished, False)

In [23]:
display(fact_sales_df.limit(10))

StatementMeta(, 23642936-a065-41bf-b59a-4ea55a1fdb36, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b1118961-9fc7-47a8-b61c-35b3a7a6359b)

In [24]:
display(
    fact_sales_df.filter(
        col("CustomerKey").isNull() |
        col("ProductKey").isNull() |
        col("StoreKey").isNull()
    )
)

StatementMeta(, 23642936-a065-41bf-b59a-4ea55a1fdb36, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ff91aae4-6d3e-4a6a-99d0-66cc36560d7b)

In [25]:
fact_sales_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_fact_sales")

StatementMeta(, 23642936-a065-41bf-b59a-4ea55a1fdb36, 27, Finished, Available, Finished, False)